# 🔷 STEP 2: DATA CLEANING
Comprehensive data quality checks and cleaning operations

In [0]:
from pyspark.sql.functions import col, sum, count, stddev, mean, min, max

# Load the raw data saved in previous notebook
engine_df = spark.read.table("cmapss_raw_data")

print("✅ Data loaded from Delta table")
print(f"Initial shape: {engine_df.count()} rows, {len(engine_df.columns)} columns")
display(engine_df.limit(5))

✅ Data loaded from Delta table
Initial shape: 160359 rows, 27 columns


engine_id,cycle,op1,op2,op3,s1,s2,s3,s4,s5,s6,s7,s8,s9,s10,s11,s12,s13,s14,s15,s16,s17,s18,s19,s20,s21,dataset
21,84,-0.0023,4.0E-4,100.0,518.67,642.26,1577.62,1396.35,14.62,21.58,554.28,2387.99,9052.67,1.3,47.2,522.65,2387.98,8139.46,8.3805,0.03,392,2388,100.0,39.12,23.4101,FD003
21,85,0.0016,-3.0E-4,100.0,518.67,642.59,1583.2,1396.73,14.62,21.58,556.2,2388.0,9055.07,1.3,47.2,522.95,2388.07,8140.44,8.3545,0.03,392,2388,100.0,39.2,23.5136,FD003
21,86,0.0019,-2.0E-4,100.0,518.67,641.99,1583.19,1400.71,14.62,21.58,554.61,2387.98,9050.32,1.3,47.18,522.04,2388.04,8136.17,8.3914,0.03,393,2388,100.0,39.19,23.3024,FD003
21,87,-0.0018,4.0E-4,100.0,518.67,641.98,1580.29,1398.98,14.62,21.58,554.64,2387.98,9052.6,1.3,47.25,523.48,2387.97,8140.67,8.3512,0.03,392,2388,100.0,39.12,23.4831,FD003
21,88,-0.0011,-2.0E-4,100.0,518.67,642.36,1577.87,1396.47,14.62,21.58,556.0,2388.04,9058.58,1.3,46.86,522.98,2388.04,8140.31,8.3583,0.03,391,2388,100.0,39.1,23.4357,FD003


## 1️⃣ Schema Validation

In [0]:
# Print schema to verify data types
engine_df.printSchema()

print("\n📊 Column Data Types Summary:")
for field in engine_df.schema.fields:
    print(f"  {field.name}: {field.dataType}")

root
 |-- engine_id: long (nullable = true)
 |-- cycle: long (nullable = true)
 |-- op1: double (nullable = true)
 |-- op2: double (nullable = true)
 |-- op3: double (nullable = true)
 |-- s1: double (nullable = true)
 |-- s2: double (nullable = true)
 |-- s3: double (nullable = true)
 |-- s4: double (nullable = true)
 |-- s5: double (nullable = true)
 |-- s6: double (nullable = true)
 |-- s7: double (nullable = true)
 |-- s8: double (nullable = true)
 |-- s9: double (nullable = true)
 |-- s10: double (nullable = true)
 |-- s11: double (nullable = true)
 |-- s12: double (nullable = true)
 |-- s13: double (nullable = true)
 |-- s14: double (nullable = true)
 |-- s15: double (nullable = true)
 |-- s16: double (nullable = true)
 |-- s17: long (nullable = true)
 |-- s18: long (nullable = true)
 |-- s19: double (nullable = true)
 |-- s20: double (nullable = true)
 |-- s21: double (nullable = true)
 |-- dataset: string (nullable = true)


📊 Column Data Types Summary:
  engine_id: LongType()


## 2️⃣ Missing Value Detection

In [0]:
# Count missing values for each column
missing_counts = engine_df.select([
    sum(col(c).isNull().cast("int")).alias(c) 
    for c in engine_df.columns
])

print("Missing values per column:")
missing_df = missing_counts.toPandas().T
missing_df.columns = ['Missing_Count']
missing_df['Missing_Percentage'] = (missing_df['Missing_Count'] / engine_df.count()) * 100
print(missing_df[missing_df['Missing_Count'] > 0])

total_missing = missing_df['Missing_Count'].sum()
if total_missing == 0:
    print("\n✅ No missing values detected!")
else:
    print(f"\n⚠️  Total missing values: {total_missing}")

Missing values per column:
Empty DataFrame
Columns: [Missing_Count, Missing_Percentage]
Index: []

✅ No missing values detected!


## 3️⃣ Handle Missing Values

In [0]:
# Drop rows with any null values
initial_count = engine_df.count()
engine_df = engine_df.dropna()
final_count = engine_df.count()

print(f"Rows before: {initial_count}")
print(f"Rows after: {final_count}")
print(f"Rows removed: {initial_count - final_count}")
print("✅ Missing value handling complete")

Rows before: 160359
Rows after: 160359
Rows removed: 0
✅ Missing value handling complete


## 4️⃣ Duplicate Detection and Removal

In [0]:
# Check and remove duplicates
initial_count = engine_df.count()
engine_df = engine_df.dropDuplicates()
final_count = engine_df.count()

print(f"Rows before: {initial_count}")
print(f"Rows after: {final_count}")
print(f"Duplicates removed: {initial_count - final_count}")
print("✅ Duplicate removal complete")

Rows before: 160359
Rows after: 160359
Duplicates removed: 0
✅ Duplicate removal complete


## 5️⃣ Constant/Low Variance Column Detection

In [0]:
# Calculate standard deviation for all numeric columns
numeric_cols = [f.name for f in engine_df.schema.fields 
                if str(f.dataType) in ['DoubleType()', 'IntegerType()', 'LongType()']]

print("Standard deviation for each numeric column:\n")
variance_stats = []

for c in numeric_cols:
    if c not in ['engine_id', 'cycle']:  # Skip ID and cycle columns
        std_val = engine_df.select(stddev(c)).first()[0]
        variance_stats.append({'column': c, 'std_dev': std_val if std_val else 0})

import pandas as pd
variance_df = pd.DataFrame(variance_stats).sort_values('std_dev')
print(variance_df.to_string(index=False))

# Identify low variance columns (std < 0.01)
low_variance_cols = variance_df[variance_df['std_dev'] < 0.01]['column'].tolist()
print(f"\n⚠️  Low variance columns (std < 0.01): {low_variance_cols}")

Standard deviation for each numeric column:

column    std_dev
   s16   0.004997
   s10   0.142103
   op2   0.367938
   s15   0.751581
   s11   3.426342
    s5   4.265554
   s19   4.656270
    s6   6.443922
   s21   7.015067
   s20  11.691422
   op3  12.359044
   op1  16.527988
    s1  30.420388
   s17  31.021430
    s2  42.478516
   s14  80.623257
   s13 111.167242
    s3 118.175261
    s4 136.300073
    s8 142.426613
   s18 142.513114
   s12 164.193480
    s7 174.133835
    s9 374.657454

⚠️  Low variance columns (std < 0.01): ['s16']


## 6️⃣ Remove Constant Columns

In [0]:
# Drop low variance columns
if low_variance_cols:
    engine_df = engine_df.drop(*low_variance_cols)
    print(f"✅ Removed {len(low_variance_cols)} low variance columns")
    print(f"Removed columns: {low_variance_cols}")
else:
    print("✅ No low variance columns to remove")

print(f"\nCurrent shape: {engine_df.count()} rows, {len(engine_df.columns)} columns")

✅ Removed 1 low variance columns
Removed columns: ['s16']

Current shape: 160359 rows, 26 columns


## 7️⃣ Outlier Detection (IQR Method)

In [0]:
# Sample data for outlier analysis (use 20% sample for efficiency)
sample_df = engine_df.sample(0.2, seed=42)
pdf = sample_df.toPandas()

print(f"Analyzing outliers on sample: {len(pdf)} rows\n")

# Calculate IQR for key sensor columns
sensor_cols = [c for c in pdf.columns if c.startswith('s')]

outlier_summary = []
for col in sensor_cols[:5]:  # Check first 5 sensors
    Q1 = pdf[col].quantile(0.25)
    Q3 = pdf[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = pdf[(pdf[col] < lower_bound) | (pdf[col] > upper_bound)]
    outlier_pct = (len(outliers) / len(pdf)) * 100
    
    outlier_summary.append({
        'Sensor': col,
        'Q1': Q1,
        'Q3': Q3,
        'IQR': IQR,
        'Lower': lower_bound,
        'Upper': upper_bound,
        'Outlier_Count': len(outliers),
        'Outlier_%': outlier_pct
    })

import pandas as pd
outlier_df = pd.DataFrame(outlier_summary)
print(outlier_df.to_string(index=False))

print("\n📝 Note: Outlier removal is optional and should be done carefully")
print("For predictive maintenance, extreme values may indicate actual degradation")

Analyzing outliers on sample: 32009 rows

Sensor      Q1      Q3    IQR    Lower    Upper  Outlier_Count  Outlier_%
    s1  449.44  518.67  69.23  345.595  622.515              0        0.0
    s2  549.95  642.35  92.40  411.350  780.950              0        0.0
    s3 1357.23 1586.67 229.44 1013.070 1930.830              0        0.0
    s4 1126.88 1402.17 275.29  713.945 1815.105              0        0.0
    s5    5.48   14.62   9.14   -8.230   28.330              0        0.0

📝 Note: Outlier removal is optional and should be done carefully
For predictive maintenance, extreme values may indicate actual degradation


## 8️⃣ Save Cleaned Data

In [0]:
# Save cleaned data as new Delta table
engine_df.write.format("delta").mode("overwrite").saveAsTable("cmapss_cleaned_data")

print("✅ Cleaned data saved as Delta table: cmapss_cleaned_data")
print(f"\nFinal shape: {engine_df.count()} rows, {len(engine_df.columns)} columns")
print(f"Columns: {engine_df.columns}")

# Display sample of cleaned data
display(engine_df.limit(10))

✅ Cleaned data saved as Delta table: cmapss_cleaned_data

Final shape: 160359 rows, 26 columns
Columns: ['engine_id', 'cycle', 'op1', 'op2', 'op3', 's1', 's2', 's3', 's4', 's5', 's6', 's7', 's8', 's9', 's10', 's11', 's12', 's13', 's14', 's15', 's17', 's18', 's19', 's20', 's21', 'dataset']


engine_id,cycle,op1,op2,op3,s1,s2,s3,s4,s5,s6,s7,s8,s9,s10,s11,s12,s13,s14,s15,s17,s18,s19,s20,s21,dataset
87,58,19.9981,0.7004,100.0,491.19,607.01,1484.67,1250.9,9.35,13.65,336.5,2324.02,8725.95,1.08,44.18,316.53,2388.1,8066.86,9.1512,364,2324,100.0,24.67,14.7624,FD004
87,60,10.0011,0.25,100.0,489.05,604.73,1494.44,1295.78,10.52,15.49,396.56,2318.87,8784.51,1.26,45.11,373.31,2388.17,8137.21,8.5954,370,2319,100.0,28.78,17.2833,FD004
87,61,10.0011,0.2512,100.0,489.05,603.8,1497.48,1299.05,10.52,15.48,396.71,2318.91,8779.23,1.26,45.16,373.26,2388.09,8140.73,8.558,368,2319,100.0,28.52,17.1565,FD004
87,67,10.0043,0.25,100.0,489.05,604.85,1497.94,1300.48,10.52,15.49,397.03,2318.91,8782.35,1.26,45.35,373.75,2388.09,8132.87,8.5688,367,2319,100.0,28.78,17.3155,FD004
87,74,20.0045,0.7019,100.0,491.19,607.0,1479.33,1251.85,9.35,13.65,337.22,2324.01,8739.29,1.08,44.32,317.05,2388.21,8071.23,9.1357,364,2324,100.0,24.57,14.7025,FD004
87,89,35.0031,0.84,100.0,449.44,555.31,1362.56,1128.12,5.48,8.0,196.48,2223.06,8358.01,1.03,41.59,184.75,2388.13,8076.3,9.2399,332,2223,100.0,14.99,9.0472,FD004
87,111,41.9991,0.8413,100.0,445.0,549.44,1348.89,1115.99,3.91,5.71,138.99,2212.07,8330.11,1.03,41.81,131.8,2388.07,8093.84,9.2971,329,2212,100.0,10.7,6.4753,FD004
87,118,0.0022,0.0014,100.0,518.67,641.86,1583.28,1401.74,14.62,21.6,556.73,2388.07,9066.44,1.3,47.29,525.02,2388.16,8145.09,8.3777,393,2388,100.0,39.23,23.3695,FD004
87,119,35.0007,0.8401,100.0,449.44,555.28,1368.15,1126.39,5.48,7.99,195.83,2223.0,8355.64,1.03,41.77,184.69,2388.15,8081.62,9.2453,332,2223,100.0,14.96,8.9472,FD004
87,176,24.9988,0.6211,60.0,462.54,536.81,1264.63,1040.12,7.05,9.02,177.49,1915.5,8022.29,0.94,36.55,166.16,2028.43,7888.08,10.7808,308,1915,84.93,14.36,8.6172,FD004


## 📊 Data Cleaning Summary
✅ Schema validated  
✅ Missing values handled  
✅ Duplicates removed  
✅ Low variance columns removed  
✅ Outliers analyzed  
✅ Clean data saved as: `cmapss_cleaned_data`

**Next Step**: Proceed to `03_EDA` notebook for exploratory analysis